# 00 — Dataset layout

This is the first of **five** tutorial notebooks for the HCD
(high-column-density absorber) catalog and emulator-input dataset.  By
the end of the five notebooks you will be able to:

| Notebook | What you'll learn |
|---|---|
| 00 | Where the data lives, how the parameter space is encoded in folder names, how snapshots map to redshift, what files exist for each (sim, snap). |
| 01 | How to read the per-sim **absorber catalog** (`catalog.npz`) and the underlying **raw spectra** HDF5 file; visualise a single DLA. |
| 02 | How to read the cached **CDDF** and **dN/dX** for one (sim, snap) and how to recompute them from the catalog (and reproduce a literature comparison plot). |
| 03 | How to read the cached **per-class P1D** templates (`p1d_per_class.h5`) and how to recompute them from raw spectra + catalog using the project's own helpers. |
| 04 | A single sightline end-to-end: τ(v) → F(v), the finder algorithm stage-by-stage, fast vs Voigt NHI estimation, masking and fill strategies. |

Why this matters: the per-sim outputs in this dataset are the inputs to the
HCD emulator we are building.  Before you can train or evaluate that emulator
you need to be confident that you understand — and can reproduce — the four
observables on disk.

**Audience.** A new graduate student or RA picking up this project for the
first time.  We assume comfort with numpy / matplotlib / h5py.  We do **not**
assume prior exposure to the Lyα forest, to this codebase, or to
emulator-building.  The primer immediately below establishes the
physical picture; if you already know the field, skim it and skip to
the glossary in §0.


## What is the Lyα forest? (and what is this project trying to do?)

A six-bullet primer for a first-day grad student.

* **The forest as a transmission spectrum.**  Light from a distant
  quasar passes through the intergalactic medium (IGM) on its way to
  us.  Wherever there's neutral hydrogen along the line of sight, the
  Lyα transition (rest wavelength 1215.67 Å) absorbs a piece of the
  spectrum.  What we observe is therefore not the IGM itself but a
  *transmission spectrum* `F(λ) = exp(−τ(λ))`, where `τ` is the
  velocity-integrated optical depth.  The "forest" is the dense forest
  of overlapping low-τ absorption features that this transmission
  spectrum acquires between the quasar's Lyα and Lyβ rest lines.

* **Two phases of the IGM.**  Most pixels in the forest are *optically
  thin* — they trace small density fluctuations of warm, photoionised
  hydrogen, and `τ` for a given pixel scales roughly as `(1+δ_b)^α` at
  fixed photoionisation rate.  A *small* fraction of sightlines pass
  through self-shielded, dense, cold gas — the **HCDs**: Lyman-Limit
  Systems (LLS), sub-DLAs, and Damped Lyα Absorbers (DLAs), in
  ascending order of `N_HI`.  These are the rare, optically-thick tail
  on the same continuous distribution of column densities.

* **Why P1D is the canonical observable.**  Observational surveys
  (BOSS, eBOSS, DESI, HiRes) report `⟨F⟩(z)` and the *1D flux power
  spectrum* `P_{1D}(k, z)` measured along each sightline, because 3D
  forest statistics require quasar pair sampling that's only just
  becoming feasible.  The 1D power spectrum is what cosmology
  constraints from the Lyα forest are based on, and it's what the
  downstream HCD emulator in this project will need to reproduce.

* **The HCD problem in one sentence.**  A DLA on a sightline lowers
  that sightline's `⟨F⟩` and adds correlated power at low `k` that
  *masquerades as cosmology* — leaving it in biases inferred parameters.
  This project's job is to separate the HCD-induced contribution from
  the forest signal, so that downstream analyses can either mask HCDs
  out or marginalise over a per-class template.

* **Map from physics to observables to notebooks.**  Four observables
  drop out of this picture:
    1. The **absorber catalog** — one row per HCD with `(N_HI, b, v_0)` — is
       what NB01 walks through.
    2. The **CDDF** `f(N_HI)` and its integral **dN/dX** — number
       statistics of HCDs as a function of column density — are NB02.
    3. The **per-class P1D** templates `(P_clean, P_LLS_only,
       P_subDLA_only, P_DLA_only)` — *what HCDs add to the forest power
       spectrum, decomposed by class* — are NB03.
    4. The **per-spectrum view** — masking and fill strategies on a
       single sightline — is NB04.

* **Reading suggestion before continuing.**  If anything above is
  unfamiliar, skim §1–3 of Croft+1998 or §1–3 of Wolfe, Gawiser &
  Prochaska 2005 before you start NB01.  Both are linked from
  `notebooks/tutorials/README.md` under "Background reading".


## 0. Glossary of jargon used throughout

If any of the below is unfamiliar, this is a quick reference; the
notebooks define them more carefully on first use.

| Term | One-line definition |
|---|---|
| **τ** (tau) | Optical depth along a sightline at a given velocity offset; `F = exp(-τ)` is the transmitted flux fraction.  Dimensionless. |
| **F**, ⟨F⟩ | Transmitted flux fraction `F = exp(-τ)` (per pixel); ⟨F⟩ is its mean over a sample of sightlines. |
| **δF** | Flux contrast `δF = F/⟨F⟩ − 1`; mean-zero quantity used for power spectra. |
| **NHI** | Neutral-hydrogen column density of an absorber, in cm⁻².  Standard plot axis is `log10(N_HI)`. |
| **LLS / subDLA / DLA** | HCD classes by `log10(N_HI)`: [17.2, 19.0), [19.0, 20.3), ≥ 20.3 respectively. |
| **Damping wings** | Lorentzian tails of the Voigt profile that dominate τ for log NHI ≳ 20 absorbers; carry most of the column-density information for DLAs. |
| **CDDF** | Column-density distribution function `f(N_HI) = d²n / (dN_HI · dX)`; units cm². |
| **dX (absorption distance)** | Cosmology-weighted path length, `dX = (1+z)² · (H₀/H(z)) · dz`.  Using `dX` rather than `dz` removes most of the explicit z-dependence from `f(N_HI)`. |
| **dN/dX** | Per-class HCD incidence rate, `dN/dX |_class = ∫_class f(N_HI) · dN_HI`.  Dimensionless. |
| **P1D(k)** | 1D flux power spectrum, P1D = ⟨|FFT[δF]|²⟩ · dv / N.  Units of velocity (km/s) when `k` is in s/km. |
| **Cyclic vs angular k** | Cyclic: `k_cyc = m / L` (units s/km if L in km/s); angular: `k_ang = 2π k_cyc` (rad·s/km).  The HDF5 files store `k_cyc` (from `numpy.fft.rfftfreq`); PRIYA emulator plots use `k_ang`. |
| **Rogers per-class P1D** | Four-template decomposition of the P1D into `P_clean`, `P_LLS_only`, `P_subDLA_only`, `P_DLA_only`, each normalised by its **own subset's** mean flux.  Used as a parametric HCD-bias template downstream. |
| **fast-mode NHI** vs **Voigt NHI** | Two ways to turn τ(v) into a column density.  The **fast** estimator applies the Ladenburg–Reiche sum rule directly to the τ array (`N_HI = Σ τ_i dv / σ_int`); this is *exact* for noise-free, complete profiles — the only error source is truncation of pixels below `τ_threshold`, which is `< 0.005 dex` worst case (full derivation in `docs/fast_mode_physics.md`).  The **Voigt** fit fits a single-component Voigt profile on a widened window; it agrees with fast mode to `< 0.001 dex` on clean profiles, but gives you `b` for free and is more robust to a single contaminating absorber inside the fit window.  Fast mode is the production LF setting; Voigt is the HR setting (HR has the SNR to use `b`). |
| **`(1+z)·h` bug** | Historical absorption-distance bug: the original `cddf.npz` was **inflated** by a missing `(1+z)·h` factor in the path-length denominator.  The corrected file is the original divided by `patch_factor = (1+z)·h` (≈ 2.0 at z=2, h=0.67).  Always use `cddf_corrected.npz`. |


## 1. The two storage locations

The data this project uses lives in two places, and they have very
different sizes:

| Path | Contents | Per-file size |
|---|---|---|
| `/nfs/turbo/umor-yueyingn/mfho/emu_full/<sim>/output/SPECTRA_NNN/lya_forest_spectra_grid_480.hdf5` | **Raw** Lyman-α optical-depth skewers (`tau`), 691 200 sightlines on a regular 480² grid × the LOS axis | ~3 GB |
| `/scratch/cavestru_root/cavestru0/mfho/hcd_outputs/<sim>/snap_NNN/` | **Processed** per-(sim, snap) outputs — absorber catalog, CDDF, P1D variants, per-class P1D, metadata | ~few MB total |

A few rules to internalise:

* The **raw** files are 3 GB each and there are ~1000 of them — never
  copy them, never load a full file into memory, always stream them in
  batches (we'll show how in notebook 01).
* The **processed** files are tiny — small enough that the whole 1076-row
  dataset fits in memory once aggregated.  These are what the emulator
  trains on.
* Both locations are read-only from your point of view as a student.
  Generate any new outputs into a fresh directory under your scratch.


In [1]:
# Imports used throughout this notebook
from pathlib import Path
import numpy as np

DATA_ROOT = Path('/nfs/turbo/umor-yueyingn/mfho/emu_full')
HCD_OUT_ROOT = Path('/scratch/cavestru_root/cavestru0/mfho/hcd_outputs')

print('raw spectra root :', DATA_ROOT,  '->', DATA_ROOT.exists())
print('processed root   :', HCD_OUT_ROOT, '->', HCD_OUT_ROOT.exists())


raw spectra root : /nfs/turbo/umor-yueyingn/mfho/emu_full -> True
processed root   : /scratch/cavestru_root/cavestru0/mfho/hcd_outputs -> True


## 2. Simulation folder names encode the parameter point

Each simulation directory under `HCD_OUT_ROOT` is named like:

```
ns0.81Ap1.6e-09herei3.59heref2.79alphaq1.71hub0.668omegamh20.145hireionz7.92bhfeedback0.0333
```

The nine parameters that appear in that name are the dimensions of the
emulator parameter space:

| Symbol | Meaning | Approx range |
|---|---|---|
| `ns` | spectral index of primordial power | 0.80 – 1.05 |
| `Ap` | amplitude `A_s` (10⁻⁹) | 1.2e-9 – 2.6e-9 |
| `herei` | start z of HeII reionisation | 3.5 – 4.5 |
| `heref` | end z of HeII reionisation | 2.6 – 3.2 |
| `alphaq` | quasar SED spectral index | 1.3 – 3.0 |
| `hub` | dimensionless Hubble `h` | 0.65 – 0.75 |
| `omegamh2` | matter density `Ω_m h²` | 0.140 – 0.146 |
| `hireionz` | start z of HI reionisation | 6.5 – 8.0 |
| `bhfeedback` | black-hole feedback amplitude | 0.03 – 0.07 |

The package gives you a parser so you don't have to write a regex by hand:


In [2]:
from hcd_analysis.io import parse_sim_params, discover_simulations

example = 'ns0.81Ap1.6e-09herei3.59heref2.79alphaq1.71hub0.668omegamh20.145hireionz7.92bhfeedback0.0333'
params = parse_sim_params(example)
for k, v in params.items():
    print(f'  {k:12s} = {v}')


  ns           = 0.81
  Ap           = 1.6e-09
  herei        = 3.59
  heref        = 2.79
  alphaq       = 1.71
  hub          = 0.668
  omegamh2     = 0.145
  hireionz     = 7.92
  bhfeedback   = 0.0333


## 3. The full simulation list

`discover_simulations()` walks a directory and returns every folder whose
name matches the 9-parameter pattern.  Use it on `HCD_OUT_ROOT` to see all
LF (low-fidelity, 60 sims) entries; the 4 HR (high-fidelity) sims live in
the `hires/` subdirectory.


In [3]:
sims_lf = discover_simulations(HCD_OUT_ROOT)
sims_hr = discover_simulations(HCD_OUT_ROOT / 'hires')

print(f'LF sims discovered: {len(sims_lf)}')
print(f'HR sims discovered: {len(sims_hr)}')

# Peek at the first three so you can see the SimInfo structure
for s in sims_lf[:3]:
    print(s.name[:60], ' -> ns=%.3f Ap=%.2e' % (s.params['ns'], s.params['Ap']))


LF sims discovered: 60
HR sims discovered: 4
ns0.803Ap2.2e-09herei4.05heref2.67alphaq2.21hub0.735omegamh2  -> ns=0.803 Ap=2.20e-09
ns0.813Ap2.51e-09herei3.84heref3.02alphaq1.66hub0.68omegamh2  -> ns=0.813 Ap=2.51e-09
ns0.816Ap1.55e-09herei3.55heref3.15alphaq1.48hub0.658omegamh  -> ns=0.816 Ap=1.55e-09


## 4. Snapshot ↔ redshift mapping

Each simulation writes ~20 snapshots between z = 6 and z = 2.  The mapping
from snapshot index to scale factor `a` lives in
`<sim>/output/Snapshots.txt`; redshift is `z = 1/a − 1`.

The canonical-numbering table below maps snap index → redshift for
sims that have *every* snapshot output:

| Snap | z    |  | Snap | z    |
|------|------|--|------|------|
| 004  | 5.40 |  | 014  | 3.60 |
| 005  | 5.20 |  | 015  | 3.40 |
| 006  | 5.00 |  | 016  | 3.20 |
| 007  | 4.80 |  | 017  | 3.00 |
| 008  | 4.60 |  | 018  | 2.80 |
| 009  | 4.40 |  | 019  | 2.60 |
| 010  | 4.20 |  | 020  | 2.56 |
| 011  | 4.00 |  | 021  | 2.40 |
| 012  | 3.80 |  | 022  | 2.20 |
| 013  | 3.74 |  | 023  | 2.00 |

**Three practical caveats — read these:**

1. **Sims can have different snap-z mappings.**  Each sim has its own
   `Snapshots.txt`, and sims that were stopped early or that skipped
   intermediate outputs end up with snaps that look like the table
   above but point to a different scale factor.  Concretely: the
   example sim used throughout this tutorial has its `snap_022` at
   **z = 2.0** (not 2.2), because that sim's output schedule skipped
   one of the higher-z snaps.  **`meta.json["z"]` is the authoritative
   z for any (sim, snap) — always read it; never rely on snap
   number alone.**
2. Not every sim has every snapshot.  Snap 022 (canonical z = 2.2) and
   snap 023 (canonical z = 2.0) are missing in some sims that weren't
   run all the way down.
3. We ignore snap 016 (canonical z = 3.20) for some analyses where it
   overlaps awkwardly with neighbouring snaps; see the analysis docs.


In [4]:
# The authoritative per-(sim, snap) redshift is meta.json["z"] in each
# processed-output snap directory.  Two ways to recover the full snap-z
# mapping for one sim:
#
#   (1) Read each snap_NNN/meta.json directly (works for every sim that has
#       any processed outputs at all — this is the path we use here).
#   (2) Read the raw-data side's output/Snapshots.txt via
#       hcd_analysis.io.read_snapshots_txt(sim_info).  This is the original
#       source of truth but is missing for some sims (early-stopped runs,
#       or sims whose Snapshots.txt was never written).
import json

sim_dir = HCD_OUT_ROOT / example
snap_dirs = sorted(p for p in sim_dir.iterdir()
                   if p.is_dir() and p.name.startswith('snap_'))

print(f'{len(snap_dirs)} snapshots present on disk for this sim:')
print(f'{"snap":>4}  {"a":>8}  {"z":>6}')
for sd in snap_dirs:
    with open(sd / 'meta.json') as f:
        m = json.load(f)
    z = float(m['z'])
    a = 1.0 / (1.0 + z)
    snap_num = int(m['snap'])
    print(f'{snap_num:>4d}  {a:>8.4f}  {z:>6.3f}')

# Sanity-check the snap→z mapping against the canonical table at the top of
# this notebook.  Any sim that skipped a canonical snap will show a
# mismatch; that's expected (the table is the "fully run" mapping).


18 snapshots present on disk for this sim:
snap         a       z
   4    0.1562   5.400
   5    0.1613   5.200
   6    0.1667   5.000
   7    0.1724   4.800
   8    0.1786   4.600
   9    0.1852   4.400
  10    0.1923   4.200
  11    0.2000   4.000
  12    0.2083   3.800
  13    0.2174   3.600
  14    0.2273   3.400
  15    0.2381   3.200
  17    0.2500   3.000
  18    0.2632   2.800
  19    0.2778   2.600
  20    0.2941   2.400
  21    0.3125   2.200
  22    0.3333   2.000


## 5. What's inside each `<sim>/snap_NNN/` directory

For every (sim, snap) pair the pipeline writes a small set of files into
`HCD_OUT_ROOT/<sim>/snap_NNN/`.  The ones you'll touch most often:

| File | Contents | Tutorial |
|---|---|---|
| `meta.json` | Sim name, snap, z, dv_kms, n_skewers, parsed `sim_ics`, absorber counts | NB 01 |
| `catalog.npz` | Per-absorber records: `skewer_idx`, `pix_start`, `pix_end`, `NHI`, `b_kms`, `fit_success`, `fast_mode` | NB 01 |
| `cddf_corrected.npz` | Column-density distribution function on a 30-bin log10 NHI grid (the **corrected** version, after the absorption-path bug fix) | NB 02 |
| `p1d_per_class.h5` | Per-class P1D templates (`P_clean`, `P_LLS_only`, `P_subDLA_only`, `P_DLA_only`) on the sim's native rfftfreq k-grid | NB 03 |

A few less-used files — you may see them but should generally ignore unless
you have a specific reason:

* `cddf.npz` — original (buggy) CDDF before the `(1+z)·h` absorption-path
  fix.  **Always prefer `cddf_corrected.npz`.**
* `cddf_corrected.npz.tmp.npz` — stale write-then-rename artifact left
  behind by the patch script (the script writes to a `.tmp.npz` and
  renames, but a small number of snaps never finished the rename).
  Safe to ignore; contents are identical to `cddf_corrected.npz`.
* `p1d.npz`, `p1d_excl.npz`, `p1d_ratios.npz` — older / aggregate P1D
  variants superseded by `p1d_per_class.h5` for emulator inputs.
* `done` — empty marker file the pipeline writes when the (sim, snap) is
  fully processed.

Let's verify the files exist for one (sim, snap):


In [5]:
import json
SIM_DIR = HCD_OUT_ROOT / example
SNAP = 22
SNAP_DIR = SIM_DIR / f'snap_{SNAP:03d}'

print('Listing', SNAP_DIR)
for p in sorted(SNAP_DIR.iterdir()):
    print(f'  {p.name:30s}  {p.stat().st_size:>10d} B')

# Quick peek at meta.json so you know it's just a JSON
with open(SNAP_DIR / 'meta.json') as f:
    meta = json.load(f)
print()
print('meta.json keys:', list(meta.keys()))
print('  z          =', meta['z'])
print('  dv_kms     =', meta['dv_kms'])
print('  n_skewers  =', meta['n_skewers'])
print('  n_absorbers=', meta['n_absorbers'])


Listing /scratch/cavestru_root/cavestru0/mfho/hcd_outputs/ns0.81Ap1.6e-09herei3.59heref2.79alphaq1.71hub0.668omegamh20.145hireionz7.92bhfeedback0.0333/snap_022
  catalog.npz                        2455972 B
  cddf.npz                              3044 B
  cddf_corrected.npz                    3831 B
  cddf_corrected.npz.tmp.npz            3831 B
  done                                     0 B
  meta.json                              910 B
  p1d.npz                               3168 B
  p1d_excl.npz                          5914 B
  p1d_per_class.h5                     38200 B
  p1d_ratios.npz                        2006 B

meta.json keys: ['sim_name', 'snap', 'z', 'dv_kms', 'nbins', 'n_skewers', 'box_kpc_h', 'hubble', 'n_absorbers', 'timing_s', 'sim_ics']
  z          = 2.0
  dv_kms     = 10.00505191229706
  n_skewers  = 691200
  n_absorbers= {'LLS': 55070, 'subDLA': 17641, 'DLA': 9051, 'forest': 0}


## 6. One important wrinkle: the k-grid is not shared across sims

The per-class P1D `k` array on disk uses numpy's `rfftfreq` convention,
and its length is `nbins/2 + 1`.  `nbins` is the number of velocity
pixels per skewer, which depends on the box size in km/s — and that
depends on `H(z)` (cosmology-dependent) and the sim's box size.  So:

* **Different sims have different `nbins`** at the same snap (e.g. 1141,
  1228, 1268 are all real values seen in the LF set at snap 022).
* **Different snaps within one sim have different `nbins`** because
  `H(z)` changes with redshift.
* The `k` arrays therefore have different lengths across (sim, snap)
  pairs (e.g. 553–635 entries at snap 022 across 60 sims).

This matters for emulator construction: you can't just stack the on-disk
P1Ds into a `(N_sims, n_k)` matrix without first interpolating onto a
shared k-grid.  We'll come back to this in notebook 03; for now just
remember it.

**Note on k convention.**  The `k` array stored in `p1d_per_class.h5`
is the **cyclic** wavenumber from `numpy.fft.rfftfreq` (units of
`s/km`).  The project-wide convention adopted by PRIYA / Rogers / the
HCD emulator inputs is the **angular** wavenumber `k_ang = 2π · k_cyc`
(units of `rad·s/km`).  **NB03 §1b makes the switch explicit** and
every subsequent plot in NB03 — plus the in-repo cache at
`hcd_analysis/_emulator_data/observables.h5` — is in angular units.
When you read `k` directly off disk, multiply by `2π` first.


## Where to next

* **Notebook 01** — open `catalog.npz` and the raw spectra HDF5; visualise
  a single DLA in flux space.
* **Notebook 02** — recompute the CDDF and dN/dX from the catalog and
  reproduce a literature-comparison figure.
* **Notebook 03** — recompute the per-class P1D from raw spectra +
  catalog using the project helpers.
* **Notebook 04** — walk one sightline end-to-end: τ(v) / F(v) per
  class, the finder algorithm stage-by-stage, fast vs Voigt NHI, and
  the three mask × three fill matrix.

If you ever want a quick reference for any of the file schemas, run::

    python3 -c "import h5py; f = h5py.File('<path>', 'r'); f.visititems(lambda n,o: print(n,o))"

— that prints the full HDF5 tree and is the fastest way to remind yourself
what's in a file.
